In [ ]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


# Register & stance — formality scale + sentiment polarity

Two views:

1. **Stance & register heatmaps + per-file justification distribution** —
   - Stance × category (5-class polarity-split: directive / expository / **positive_evaluative** / **negative_evaluative** / dialogic, plus 1p / 2p engagement).
   - Register × category (4-class formality scale: frozen / formal / consultative / casual).
   - Per-file justification ratio boxplot + jittered strip plot.

2. **Register quantitative scatter** — TTR × Heylighen F-score (per file, sized by token count, colored by category) plus distributions of mean sentence length and dependency depth.

The polarity split makes it possible to separate praise/recommendation language (`positive_evaluative`) from criticism/error language (`negative_evaluative`) — the corpus runs ~3.2× more positive than negative.


### Stance & register heatmaps + per-file justification distribution

Two heatmaps (stance × category and register × category) plus a per-file box-strip plot of
justification ratios. Hover any cell or point for exact values.

In [ ]:
"""Stance & register heatmaps + per-file justification distribution.

Stance is now 5-class (evaluative split into positive/negative polarity).
"""

stance_long = []
stance_keys = ["directive_pct", "expository_pct",
               "positive_evaluative_pct", "negative_evaluative_pct",
               "dialogic_pct", "pronouns_2p_pct", "pronouns_1p_pct"]
register_keys = ["frozen_pct", "formal_pct", "consultative_pct", "casual_pct"]

for cat, b in by_category.items():
    s = b["metrics"]["stance"]
    for k in stance_keys:
        stance_long.append({"category": cat, "metric": k.replace("_pct", ""),
                            "value": s[k], "kind": "stance"})
    r = b["metrics"]["register"]
    for k in register_keys:
        stance_long.append({"category": cat, "metric": k.replace("_pct", ""),
                            "value": r[k], "kind": "register"})
sr_long_df = pd.DataFrame(stance_long)

heat_stance = (
    alt.Chart(sr_long_df[sr_long_df["kind"] == "stance"])
    .mark_rect()
    .encode(
        x=alt.X("metric:N",
                sort=[k.replace("_pct", "") for k in stance_keys],
                title=None,
                axis=alt.Axis(labelAngle=-30)),
        y=alt.Y("category:N", title=None),
        color=alt.Color("value:Q", scale=alt.Scale(scheme="magma", reverse=True),
                         title="% of file tokens"),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("metric:N"),
                 alt.Tooltip("value:Q", format=".3f")],
    )
    .properties(width=460, height=260,
                title="Stance × category (% of file tokens; 5-class polarity-split)")
)

heat_register = (
    alt.Chart(sr_long_df[sr_long_df["kind"] == "register"])
    .mark_rect()
    .encode(
        x=alt.X("metric:N",
                sort=[k.replace("_pct", "") for k in register_keys],
                title=None),
        y=alt.Y("category:N", title=None),
        color=alt.Color("value:Q", scale=alt.Scale(scheme="viridis"),
                         title="% of file tokens"),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("metric:N"),
                 alt.Tooltip("value:Q", format=".3f")],
    )
    .properties(width=280, height=260, title="Register × category (% of file tokens)")
)

just_box = (
    alt.Chart(alt_df)
    .mark_boxplot(extent="min-max", opacity=0.55, color="#4e79a7")
    .encode(
        x=alt.X("category:N", title=None, sort=cats),
        y=alt.Y("just_ratio:Q", title="Justification ratio per file"),
    )
    .properties(width=520, height=260,
                title="Per-file justification ratio by category")
)
just_strip = (
    alt.Chart(alt_df)
    .mark_circle(size=35, opacity=0.45, color="#e15759")
    .encode(
        x=alt.X("category:N", title=None, sort=cats),
        y=alt.Y("just_ratio:Q"),
        xOffset="jitter:Q",
        tooltip=[alt.Tooltip("path:N"),
                 alt.Tooltip("category:N"),
                 alt.Tooltip("n_tokens:Q"),
                 alt.Tooltip("just_ratio:Q", format=".2f")],
    )
    .transform_calculate(jitter="random()-0.5")
)

(heat_stance | heat_register) & (just_box + just_strip)


### Register quantitative metrics

The four register numbers — `ttr`, `mean_sent_len`, `dep_depth`, `f_score` — were computed per
file but only the four lexical register classes were plotted earlier. Two views below cover them:

- **TTR vs F-score** scatter — every file as one point, coloured by category. Files cluster
  along a single F-score band (~70-80) with TTR varying mostly by file length.
- **Sentence length & dependency depth** distributions per category (box-strip).

In [ ]:
"""Register quantitative metrics: TTR×F-score + sentence length & dep-depth distributions."""

ttr_f_chart = (
    alt.Chart(alt_df)
    .mark_circle(opacity=0.65)
    .encode(
        x=alt.X("f_score:Q", title="F-score (Heylighen & Dewaele) — higher = more formal",
                scale=alt.Scale(domain=[40, 100])),
        y=alt.Y("ttr:Q", title="Type-token ratio (lex. diversity)",
                scale=alt.Scale(domain=[0, 1])),
        size=alt.Size("n_tokens:Q", title="tokens",
                       scale=alt.Scale(range=[20, 400])),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=_cat_domain, range=_cat_range)),
        tooltip=[
            alt.Tooltip("path:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q", format=","),
            alt.Tooltip("ttr:Q", format=".3f"),
            alt.Tooltip("f_score:Q", format=".1f"),
            alt.Tooltip("mean_sent_len:Q", title="mean sent len", format=".1f"),
            alt.Tooltip("dep_depth:Q", title="dep depth", format=".2f"),
        ],
    )
    .properties(width=520, height=380, title="Per-file register: TTR × F-score")
    .interactive()
)

sent_len_box = (
    alt.Chart(alt_df)
    .mark_boxplot(extent="min-max", opacity=0.55, color="#4e79a7")
    .encode(
        x=alt.X("category:N", title=None, sort=cats),
        y=alt.Y("mean_sent_len:Q", title="Mean sentence length (tokens)"),
    )
    .properties(width=240, height=300, title="Sentence length per category")
)
sent_len_strip = (
    alt.Chart(alt_df)
    .mark_circle(size=30, opacity=0.45, color="#e15759")
    .encode(
        x=alt.X("category:N", sort=cats, title=None),
        y=alt.Y("mean_sent_len:Q"),
        xOffset="jitter:Q",
        tooltip=[alt.Tooltip("path:N"),
                 alt.Tooltip("mean_sent_len:Q", format=".1f")],
    )
    .transform_calculate(jitter="random()-0.5")
)

dep_box = (
    alt.Chart(alt_df)
    .mark_boxplot(extent="min-max", opacity=0.55, color="#59a14f")
    .encode(
        x=alt.X("category:N", title=None, sort=cats),
        y=alt.Y("dep_depth:Q", title="Mean dependency depth"),
    )
    .properties(width=240, height=300, title="Dependency depth per category")
)
dep_strip = (
    alt.Chart(alt_df)
    .mark_circle(size=30, opacity=0.45, color="#af7aa1")
    .encode(
        x=alt.X("category:N", sort=cats, title=None),
        y=alt.Y("dep_depth:Q"),
        xOffset="jitter:Q",
        tooltip=[alt.Tooltip("path:N"),
                 alt.Tooltip("dep_depth:Q", format=".2f")],
    )
    .transform_calculate(jitter="random()-0.5")
)

ttr_f_chart & ((sent_len_box + sent_len_strip) | (dep_box + dep_strip))
